## Блок 1: Инициализация и подготовка среды

Описание: подготовка рабочего пространства — импорт библиотек, настройка путей, служебные функции многоразового использования.

In [1]:
# Подблок 1.1 — Импорт библиотек
import re                       # разбор имён файлов (год/неделя)
from pathlib import Path        # удобная работа с путями
import pandas as pd             # работа с табличными данными

In [2]:
# Подблок 1.2 — Константы проекта (пути, ожидаемые параметры)
RAW_DIR = Path(r"C:\Users\irma4\Desktop\final_project_hse_eremina\data\raw")
PROCESSED_DIR = Path(r"C:\Users\irma4\Desktop\final_project_hse_eremina\data\processed")

SKIPROWS = 1                    # число строк шапки отчёта 1С перед таблицей
EXPECTED_WEEKS_PER_SEASON = 18  # ожидаемое число недель в каждом сезоне
SEASON_START_WEEK = 40          # неделя, с которой начинается новогодний сезон

# создание папки для промежуточных результатов
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Подблок 1.3 — Служебная функция разбора имени файла
# Имя файла кодирует *календарный* год и неделю (например, `2025_01W`). Сезон длится с недели 40 одного года по неделю 05 следующего, поэтому файл `2025_01W` относится к сезону 2024, а файл `2025_40W` — уже к сезону 2025.
FILENAME_PATTERN = re.compile(r"(\d{4})_(\d{2})W")

def parse_season(filename):
    """Извлечение (календарный год, номер недели, сезон) из имени файла.
    Пример: 2025_01W -> сезон 2024; 2025_40W -> сезон 2025.
    """
    match = FILENAME_PATTERN.search(filename)
    if not match:
        raise ValueError(
            f"Не удалось распознать год/неделю в имени файла: {filename}"
        )
    year, week = int(match.group(1)), int(match.group(2))
    season = year if week >= SEASON_START_WEEK else year - 1
    return year, week, season

## Блок 2: Загрузка и консолидация данных

Описание: поиск исходных файлов, чтение каждого с разметкой по сезону, объединение в единый датасет.

Вместо ручного чтения 36 файлов используется цикл — это исключает ошибки, экономит время и делает код масштабируемым. Поле `source_file` сохраняет, из какого файла пришла каждая строка, что важно для отладки.

In [4]:
# Подблок 2.1 — Проверка, что папка с данными существует
if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"Папка с выгрузками не найдена: {RAW_DIR}"
    )
print(f"Папка с данными найдена: {RAW_DIR}")

Папка с данными найдена: C:\Users\irma4\Desktop\final_project_hse_eremina\data\raw


In [5]:
# Подблок 2.2 — Список файлов, отсортированный
all_files = sorted(RAW_DIR.glob("*.xlsx"))
print(f"Найдено файлов .xlsx: {len(all_files)}")
if all_files:
    print("Примеры имён:", [f.name for f in all_files[:3]])

Найдено файлов .xlsx: 36
Примеры имён: ['2024_40W.xlsx', '2024_41W.xlsx', '2024_42W.xlsx']


In [6]:
# Подблок 2.3 — Пустой список для накопления датафреймов
list_of_dataframes = []

In [7]:
# Подблок 2.4 — Цикл чтения всех файлов с логированием и обработкой ошибок
for file_path in all_files:
    try:
        year, week, season = parse_season(file_path.name)
        df_temp = pd.read_excel(file_path, skiprows=SKIPROWS)
        # убираем случайные пробелы в именах колонок (частая проблема 1С)
        df_temp.columns = [str(c).strip() for c in df_temp.columns]
        # служебные поля для трассировки и разметки по сезонам
        df_temp["calendar_year"] = year
        df_temp["iso_week"] = week
        df_temp["season"] = season
        df_temp["source_file"] = file_path.name
        list_of_dataframes.append(df_temp)
        print(
            f"  Загружен: {file_path.name}"
            f" | строк: {len(df_temp):>7}"
            f" | сезон: {season}"
        )
    except Exception as e:
        print(f"  ОШИБКА при чтении {file_path.name}: {e}")

  Загружен: 2024_40W.xlsx | строк:   39169 | сезон: 2024
  Загружен: 2024_41W.xlsx | строк:   56989 | сезон: 2024
  Загружен: 2024_42W.xlsx | строк:   37798 | сезон: 2024
  Загружен: 2024_43W.xlsx | строк:   51059 | сезон: 2024
  Загружен: 2024_44W.xlsx | строк:   75267 | сезон: 2024
  Загружен: 2024_45W.xlsx | строк:   79149 | сезон: 2024
  Загружен: 2024_46W.xlsx | строк:   83892 | сезон: 2024
  Загружен: 2024_47W.xlsx | строк:   87353 | сезон: 2024
  Загружен: 2024_48W.xlsx | строк:   92727 | сезон: 2024
  Загружен: 2024_49W.xlsx | строк:   93803 | сезон: 2024
  Загружен: 2024_50W.xlsx | строк:   97022 | сезон: 2024
  Загружен: 2024_51W.xlsx | строк:   96951 | сезон: 2024
  Загружен: 2024_52W.xlsx | строк:   97361 | сезон: 2024
  Загружен: 2025_01W.xlsx | строк:   90777 | сезон: 2024
  Загружен: 2025_02W.xlsx | строк:   82878 | сезон: 2024
  Загружен: 2025_03W.xlsx | строк:   82335 | сезон: 2024
  Загружен: 2025_04W.xlsx | строк:   80834 | сезон: 2024
  Загружен: 2025_05W.xlsx | стр

In [8]:
# Подблок 2.5 — Объединение всех недель в единый датафрейм
df_combined = pd.concat(list_of_dataframes, ignore_index=True)
print(
    f"Объединённый датасет: {df_combined.shape[0]:,} строк,"
    f" {df_combined.shape[1]} колонок"
)

Объединённый датасет: 3,441,294 строк, 20 колонок


## Блок 3: Первичная проверка целостности

Описание: данные собраны — проверка, можно ли им доверять, *до* того как что-либо в них менять.

In [9]:
# Подблок 3.1 — Полнота недель по каждому сезону
weeks_per_season = df_combined.groupby("season")["source_file"].nunique()
for season, n in weeks_per_season.items():
    flag = "OK" if n == EXPECTED_WEEKS_PER_SEASON else "!! ПРОВЕРЬ"
    print(f"  Сезон {season}: {n} недель ({flag})")

  Сезон 2024: 18 недель (OK)
  Сезон 2025: 18 недель (OK)


In [10]:
# Подблок 3.2 — Стабильность набора колонок между файлами
col_sets = {}
for name, group in df_combined.groupby("source_file"):
    col_sets[name] = tuple(sorted(group.columns))
unique_col_sets = set(col_sets.values())

if len(unique_col_sets) == 1:
    print("OK: во всех файлах одинаковый набор колонок")
else:
    print(f"!! Обнаружено {len(unique_col_sets)} разных наборов — проверь файлы")

OK: во всех файлах одинаковый набор колонок


In [11]:
# Подблок 3.3 — Состав магазинов по сезонам (фиксация как факт из данных: состав анализируемых магазинов определяется тем, кто реально присутствует в выгрузках)
print(df_combined.groupby("season")["Магазин"].nunique().to_string())

season
2024    146
2025    172


In [12]:
# Подблок 3.4 — Пропуски в ключевых полях идентификации
key_fields = ["Магазин", "Код", "Товарная подгруппа"]
for field in key_fields:
    if field in df_combined.columns:
        print(f"  {field}: {df_combined[field].isna().sum()} пропусков")
    else:
        print(f"  !! колонка '{field}' не найдена")

  Магазин: 0 пропусков
  Код: 0 пропусков
  Товарная подгруппа: 0 пропусков


In [13]:
# Подблок 3.5 — Типы данных по колонкам (проверка - не прочитались ли числовые поля (продажи, сток) как текст, если тип `object` вместо `float64`, значит в колонке есть нечисловые символы — потребуется исправление в блоке очистки)
print(df_combined.dtypes.to_string())

Кластер                                     object
Магазин                                     object
Категория магазина                          object
Признак вместимости                         object
Критерий магазина                           object
Дата открытия                               object
Код                                         object
Артикул                                     object
Наименование                                object
Категория товара                            object
Группа                                      object
Товарная подгруппа                          object
Сток магазина (текущий) с транзитом, шт    float64
Продажи, шт                                float64
Продажи, руб. (с НДС)                      float64
Заказ сделан, шт                           float64
calendar_year                                int64
iso_week                                     int64
season                                       int64
source_file                    

In [14]:
# Подблок 3.6 — Итоговая форма датасета и первые строки
print(
    f"Строк: {df_combined.shape[0]:,}"
    f" | Колонок: {df_combined.shape[1]}"
)
df_combined.head()

Строк: 3,441,294 | Колонок: 20


,Кластер,Магазин,Категория магазина,Признак вместимости,Критерий магазина,Дата открытия,Код,Артикул,Наименование,Категория товара,Группа,Товарная подгруппа,"Сток магазина (текущий) с транзитом, шт","Продажи, шт","Продажи, руб. (с НДС)","Заказ сделан, шт",calendar_year,iso_week,season,source_file
0,4. ПОВОЛЖЬЕ,13105 Казань (Парк Хаус),D,2XS,LFL,09.12.2022,УТ-00086061,Д5022133,"Адвент календарь сладости 31 день, 465г",Продукты,Сладости,шоколад,1.0,NaN,NaN,NaN,2024,40,2024,2024_40W.xlsx
1,5. ЦЕНТР,11036 Рязань (Премьер),B,XL,LFL,13.03.2018,УТ-00086061,Д5022133,"Адвент календарь сладости 31 день, 465г",Продукты,Сладости,шоколад,1.0,NaN,NaN,NaN,2024,40,2024,2024_40W.xlsx
2,9. ЗАП. СИБИРЬ,17025 Новосибирск (Сан Сити),Top,L,LFL,06.08.2018,УТ-00086061,Д5022133,"Адвент календарь сладости 31 день, 465г",Продукты,Сладости,шоколад,1.0,NaN,NaN,NaN,2024,40,2024,2024_40W.xlsx
3,1. МОСКВА,11102 Москва (Ереван Плаза),Top,XS,LFL,04.11.2022,УТ-00086061,Д5022133,"Адвент календарь сладости 31 день, 465г",Продукты,Сладости,шоколад,1.0,NaN,NaN,NaN,2024,40,2024,2024_40W.xlsx
4,2. МО,11124 Пушкино (Пушкино Парк),D,XS,LFL,30.11.2023,УТ-00086061,Д5022133,"Адвент календарь сладости 31 день, 465г",Продукты,Сладости,шоколад,1.0,NaN,NaN,NaN,2024,40,2024,2024_40W.xlsx


## Блок 4: Очистка и преобразование

Описание: основной этап подготовки данных к анализу.

In [15]:
# Подблок 4.1 — Создание рабочей копии и переименование длинных колонок (работа на копии, чтобы в любой момент можно было вернуться к оригиналу `df_combined`, длинные имена колонок из 1С заменяются на короткие — это сделает весь последующий код компактнее и читаемее)
df = df_combined.copy()

# размер ДО очистки для сравнения в конце блока
shape_before = df.shape

# короткие имена вместо длинных
df = df.rename(columns={
    "Сток магазина (текущий) с транзитом, шт": "stock",
    "Продажи, шт":                            "sales_qty",
    "Продажи, руб. (с НДС)":                  "sales_rub",
    "Заказ сделан, шт":                        "order_qty",
    "Категория магазина":                      "store_category",
    "Категория товара":                        "product_category",
    "Признак вместимости":                     "store_size",
    "Критерий магазина":                       "store_lfl",
    "Дата открытия":                           "open_date",
    "Товарная подгруппа":                      "subgroup",
    "Группа":                                  "product_group",
    "Кластер":                                 "cluster",
    "Магазин":                                 "store",
    "Код":                                     "sku_code",
    "Артикул":                                 "article",
    "Наименование":                            "sku_name",
})

print("Колонки после переименования:")
print(list(df.columns))

Колонки после переименования:
['cluster', 'store', 'store_category', 'store_size', 'store_lfl', 'open_date', 'sku_code', 'article', 'sku_name', 'product_category', 'product_group', 'subgroup', 'stock', 'sales_qty', 'sales_rub', 'order_qty', 'calendar_year', 'iso_week', 'season', 'source_file']


In [16]:
# Подблок 4.2 — Заполнение пропусков нулями в количественных полях
qty_cols = ["stock", "sales_qty", "sales_rub", "order_qty"]

print("Пропуски ДО заполнения:")
print(df[qty_cols].isna().sum().to_string())

df[qty_cols] = df[qty_cols].fillna(0)

print("\nПропуски ПОСЛЕ заполнения:")
print(df[qty_cols].isna().sum().to_string())

Пропуски ДО заполнения:
stock         148928
sales_qty    2325076
sales_rub    2325076
order_qty    3028263

Пропуски ПОСЛЕ заполнения:
stock        0
sales_qty    0
sales_rub    0
order_qty    0


In [17]:
# Подблок 4.3 — Приведение «Дата открытия» к типу datetime
df["open_date"] = pd.to_datetime(
    df["open_date"].astype(str).str.strip(),
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

n_bad = df["open_date"].isna().sum()
print(f"Строк с некорректной/отсутствующей датой: {n_bad} из {len(df):,}")
print(f"Уникальных дат открытия: {df['open_date'].nunique()}")
print(f"Диапазон: {df['open_date'].min()} — {df['open_date'].max()}")

Строк с некорректной/отсутствующей датой: 0 из 3,441,294
Уникальных дат открытия: 163
Диапазон: 2017-10-27 00:00:00 — 2025-12-11 00:00:00


In [18]:
# Подблок 4.4 — Нормализация регистра товарной подгруппы (в справочнике 1С одна и та же подгруппа может быть записана как «Шоколад» и «шоколад» — это подтверждено на реальных данных (тестовый файл W40 vs W50), без нормализации эти записи попадут в разные строки при агрегации и исказят результат.
n_before = df["subgroup"].nunique()

df["subgroup"] = df["subgroup"].str.strip().str.lower()

n_after = df["subgroup"].nunique()
print(f"Уникальных подгрупп до нормализации: {n_before}")
print(f"Уникальных подгрупп после нормализации: {n_after}")
print(f"Схлопнулось дублей по регистру: {n_before - n_after}")

Уникальных подгрупп до нормализации: 85
Уникальных подгрупп после нормализации: 76
Схлопнулось дублей по регистру: 9


In [19]:
# Подблок 4.5 — Построение составного ключа подгруппы (имя подгруппы не уникально: например, «наборы» встречаются в 6 разных группах (Посуда, Носки, Тапочки, Свечи, Шапки, Кухня), если агрегировать только по имени, разные товары смешаются, составной ключ `Категория товара + Группа + Подгруппа` решает эту проблему)
df["product_category"] = df["product_category"].str.strip().str.lower()
df["product_group"] = df["product_group"].str.strip().str.lower()

df["subgroup_key"] = (
    df["product_category"] + " | "
    + df["product_group"] + " | "
    + df["subgroup"]
)

print(f"Уникальных составных ключей подгрупп: {df['subgroup_key'].nunique()}")
print("\nПримеры:")
print(df["subgroup_key"].drop_duplicates().sort_values().head(10).to_string(index=False))

Уникальных составных ключей подгрупп: 83

Примеры:
  игры и развлечения | антистресс игрушки | сквиши
            игры и развлечения | брелоки | брелоки
игры и развлечения | головоломки и конструкторы...
игры и развлечения | игры для маленьких | игров...
игры и развлечения | игры для маленьких | творч...
игры и развлечения | настольные игры | развлека...
         одежда и обувь | носки, колготки | наборы
  одежда и обувь | носки, колготки | носки высокие
 одежда и обувь | носки, колготки | носки махровые
одежда и обувь | носки, колготки | носки со спо...


In [20]:
# Подблок 4.6 — Проверка и удаление дубликатов (уникальная запись определяется сочетанием «магазин + SKU + сезон + неделя», если такая комбинация встречается дважды — это дубль (одна и та же строка загрузилась повторно))
dup_key = ["store", "sku_code", "season", "iso_week"]

n_total = len(df)
n_dups = df.duplicated(subset=dup_key, keep="first").sum()

print(f"Всего строк: {n_total:,}")
print(f"Дубликатов по ключу {dup_key}: {n_dups}")

if n_dups > 0:
    df = df.drop_duplicates(subset=dup_key, keep="first").reset_index(drop=True)
    print(f"После удаления: {len(df):,} строк")
else:
    print("Дубликатов нет, удалять нечего")

Всего строк: 3,441,294
Дубликатов по ключу ['store', 'sku_code', 'season', 'iso_week']: 0
Дубликатов нет, удалять нечего


In [21]:
# Подблок 4.7 — Сборка справочника магазинов из данных
store_attrs = ["store", "cluster", "store_category", "store_size",
               "store_lfl", "open_date"]

# проверка стабильности: у каждого магазина каждый атрибут должен быть одинаков во всех неделях
print("=== Проверка стабильности атрибутов магазина ===")
problems = 0
for attr in store_attrs:
    if attr == "store":
        continue
    varying = df.groupby("store")[attr].nunique()
    unstable = varying[varying > 1]
    if len(unstable) > 0:
        problems += 1
        print(f"  !! '{attr}': у {len(unstable)} магазинов значение меняется")
        example = unstable.index[0]
        vals = df.loc[df["store"] == example, attr].unique()
        print(f"     пример — {example}: {vals}")
    else:
        print(f"  OK '{attr}': стабилен у всех магазинов")

if problems == 0:
    print("\nВсе атрибуты стабильны — справочник строится напрямую.")

=== Проверка стабильности атрибутов магазина ===
  OK 'cluster': стабилен у всех магазинов
  OK 'store_category': стабилен у всех магазинов
  OK 'store_size': стабилен у всех магазинов
  OK 'store_lfl': стабилен у всех магазинов
  OK 'open_date': стабилен у всех магазинов

Все атрибуты стабильны — справочник строится напрямую.


In [22]:
store_ref = (
    df[store_attrs]
    .drop_duplicates(subset=["store"])
    .sort_values("store")
    .reset_index(drop=True)
)

print(f"Справочник магазинов: {len(store_ref)} строк")
store_ref.head(10)

Справочник магазинов: 172 строк


,store,cluster,store_category,store_size,store_lfl,open_date
0,11003 Москва (Атриум),1. МОСКВА,Top,XS,LFL,2017-10-27
1,11004 Москва (Аэропорт),1. МОСКВА,B,XS,LFL,2017-10-27
2,11006 Москва (Коламбус),1. МОСКВА,Top,XL,LFL,2017-11-01
3,11007 Ржавки (Зеленопарк),2. МО,Top,XL,LFL,2017-11-15
4,11008 Москва (Европейский),1. МОСКВА,Top,2XS,LFL,2017-11-27
5,11009 Мытищи (Красный Кит),2. МО,C,S,LFL,2017-11-22
6,11012 Москва (Афимолл),1. МОСКВА,Top+,L,LFL,2017-12-03
7,11013 Москва (Ладья),1. МОСКВА,A,XS,LFL,2017-12-12
8,11014 Москва (Европолис),1. МОСКВА,Top,M,LFL,2017-12-16
9,11015 Сергиев Посад (Капитолий),2. МО,B,XS,LFL,2017-12-23


In [23]:
# Подблок 4.8 — Итоговая сводка по очистке (сравнение формы датасета до и после преобразований, фиксация изменений)
shape_after = df.shape

print("=== Итоги Блока 4: Очистка и преобразование ===")
print(f"Строк до очистки:  {shape_before[0]:,}")
print(f"Строк после:       {shape_after[0]:,}")
print(f"Удалено дубликатов: {shape_before[0] - shape_after[0]:,}")
print(f"Колонок до:  {shape_before[1]} | после: {shape_after[1]}")
print(f"  (добавлен составной ключ subgroup_key)")
print(f"\nУникальных магазинов: {df['store'].nunique()}")
print(f"Уникальных SKU:       {df['sku_code'].nunique()}")
print(f"Уникальных подгрупп:  {df['subgroup_key'].nunique()}")
print(f"Сезонов:              {sorted(df['season'].unique())}")

=== Итоги Блока 4: Очистка и преобразование ===
Строк до очистки:  3,441,294
Строк после:       3,441,294
Удалено дубликатов: 0
Колонок до:  20 | после: 21
  (добавлен составной ключ subgroup_key)

Уникальных магазинов: 172
Уникальных SKU:       1572
Уникальных подгрупп:  83
Сезонов:              [np.int64(2024), np.int64(2025)]


**Выводы по Блоку 4:**

Что сделано и почему:
- **Переименование колонок** — для читаемости кода (не влияет на данные).
- **fillna(0)** — пустые значения в количественных полях означают ноль события, подтверждено сверкой.
- **Нормализация регистра подгрупп** — «Шоколад» и «шоколад» схлопнуты в одну запись; без этого агрегация дала бы задвоение.
- **Составной ключ подгруппы** — имя подгруппы не уникально (например, «наборы» встречаются в 6 группах); ключ `категория | группа | подгруппа` обеспечивает однозначность.
- **Приведение даты открытия** — для последующего расчёта возраста магазина.
- **Справочник магазинов** — собран из данных, без внешнего файла; стабильность атрибутов проверена.

## Блок 5: Конструирование признаков и агрегация

Описание: создание новых расчётных полей (дефицит, активность пары, доля дефицита, возраст магазина) и свёртка данных с уровня SKU до уровня товарной подгруппы.

In [24]:
# Подблок 5.1 — Правило `ever_active`: отделяем ассортимент от «не в матрице»
# Суммирование stock + sales + orders по каждой паре (магазин, SKU, сезон).
# Если сумма > 0 хотя бы по одному полю — пара была в ассортименте.
activity = (
    df.groupby(["store", "sku_code", "season"])[["stock", "sales_qty", "order_qty"]]
    .sum()
)
activity["ever_active"] = (activity > 0).any(axis=1)

df = df.merge(
    activity["ever_active"].reset_index(),
    on=["store", "sku_code", "season"],
    how="left"
)

total_pairs = df.groupby(["store", "sku_code", "season"]).ngroups
active_pairs = df.loc[df["ever_active"]].groupby(["store", "sku_code", "season"]).ngroups
inactive_pairs = total_pairs - active_pairs

print(f"Всего пар (магазин × SKU × сезон): {total_pairs:,}")
print(f"  активных (были в ассортименте): {active_pairs:,} ({100*active_pairs/total_pairs:.1f}%)")
print(f"  неактивных (вероятно, не в матрице): {inactive_pairs:,} ({100*inactive_pairs/total_pairs:.1f}%)")

Всего пар (магазин × SKU × сезон): 256,470
  активных (были в ассортименте): 256,325 (99.9%)
  неактивных (вероятно, не в матрице): 145 (0.1%)


In [25]:
# Подблок 5.2 — Флаг дефицита `is_stockout` (дефицит фиксируется, когда на конец недели сток равен нулю у активной пары. Это означает: товар был в ассортименте магазина, но закончился. Неактивные пары из расчёта исключены — у них нулевой сток означает «товара и не должно было быть», а не дефицит)
df["is_stockout"] = (df["stock"] == 0) & df["ever_active"]

n_stockout = df["is_stockout"].sum()
n_active_rows = df["ever_active"].sum()
print(f"Строк с дефицитом (сток = 0 у активной пары): {n_stockout:,}")
print(f"Доля дефицитных наблюдений среди активных: {100*n_stockout/n_active_rows:.1f}%")

Строк с дефицитом (сток = 0 у активной пары): 148,782
Доля дефицитных наблюдений среди активных: 4.3%


In [26]:
# Подблок 5.3 — Разделение периода на функциональные окна
# Каждая неделя получает метку роли в анализе:
# - october (W40–W44) — подготовительный период; используется для расчёта коэффициента роста, но не входит в целевую переменную прогноза;
# - season (W45–W52) — основной сезон продаж (ноябрь–декабрь), целевое окно прогноза;
# - january (W01–W05) — период распродаж; используется для оценки неликвида, но не входит в целевую переменную, т.к. январские продажи отражают эффективность уценки, а не спрос на новогодний товар.
def assign_period(week):
    """Определяет роль недели в анализе."""
    if 40 <= week <= 44:
        return "october"
    elif 45 <= week <= 52:
        return "season"
    else:  # W01–W05
        return "january"

df["period"] = df["iso_week"].apply(assign_period)
print("Распределение строк по периодам:")
print(df["period"].value_counts().to_string())

Распределение строк по периодам:
period
season     1786885
january     966802
october     687607


In [27]:
# Подблок 5.4 — Агрегация с уровня SKU до уровня «магазин × подгруппа × неделя»
# Единица анализа в проекте — товарная подгруппа, а не отдельный артикул. Причины:
# 1. Решение о распределении принимается по товарной матрице, а не по конкретному SKU.
# 2. Ассортимент SKU обновляется между сезонами почти полностью, а подгруппы сопоставимы — без агрегации сравнение двух сезонов невозможно.
# 3. На уровне SKU данные разрежены (большинство ячеек — нули), на уровне подгруппы оценки статистически устойчивы.
# При агрегации сохраняется информация о дефиците: считается доля активных SKU подгруппы с нулевым стоком.
df_week = (
    df[df["ever_active"]].groupby(
        ["store", "subgroup_key", "season", "iso_week", "period",
         "cluster", "store_category", "store_size", "store_lfl", "open_date"],
        as_index=False
    )
    .agg(
        sales_qty=("sales_qty", "sum"),
        sales_rub=("sales_rub", "sum"),
        stock=("stock", "sum"),
        order_qty=("order_qty", "sum"),
        n_sku_active=("ever_active", "sum"),         # сколько SKU активно
        n_sku_stockout=("is_stockout", "sum"),       # сколько из них в дефиците
    )
)

# доля SKU подгруппы с дефицитом на этой неделе
df_week["stockout_rate"] = df_week["n_sku_stockout"] / df_week["n_sku_active"]

print(f"Агрегированный датасет (неделя): {df_week.shape[0]:,} строк")
print(f"Уникальных магазинов: {df_week['store'].nunique()}")
print(f"Уникальных подгрупп: {df_week['subgroup_key'].nunique()}")
df_week.head()

Агрегированный датасет (неделя): 392,351 строк
Уникальных магазинов: 172
Уникальных подгрупп: 83


,store,subgroup_key,season,iso_week,period,cluster,store_category,store_size,store_lfl,open_date,sales_qty,sales_rub,stock,order_qty,n_sku_active,n_sku_stockout,stockout_rate
0,11003 Москва (Атриум),игры и развлечения | антистресс игрушки | сквиши,2024,1,january,1. МОСКВА,Top,XS,LFL,2017-10-27,4.0,696.0,10.0,0.0,2,0,0.0
1,11003 Москва (Атриум),игры и развлечения | антистресс игрушки | сквиши,2024,2,january,1. МОСКВА,Top,XS,LFL,2017-10-27,1.0,99.0,9.0,0.0,2,0,0.0
2,11003 Москва (Атриум),игры и развлечения | антистресс игрушки | сквиши,2024,3,january,1. МОСКВА,Top,XS,LFL,2017-10-27,0.0,0.0,9.0,0.0,2,0,0.0
3,11003 Москва (Атриум),игры и развлечения | антистресс игрушки | сквиши,2024,4,january,1. МОСКВА,Top,XS,LFL,2017-10-27,2.0,198.0,3.0,0.0,2,1,0.5
4,11003 Москва (Атриум),игры и развлечения | антистресс игрушки | сквиши,2024,5,january,1. МОСКВА,Top,XS,LFL,2017-10-27,1.0,199.0,2.0,0.0,1,0,0.0


In [28]:
# Подблок 5.5 — Агрегация до уровня «магазин × подгруппа × сезон» (целевая переменная прогноза — суммарные продажи подгруппы в магазине за основной сезон (ноябрь–декабрь, период `season`), октябрь и январь в целевую переменную не входят)
df_season = (
    df_week[df_week["period"] == "season"].groupby(
        ["store", "subgroup_key", "season",
         "cluster", "store_category", "store_size", "store_lfl", "open_date"],
        as_index=False
    )
    .agg(
        sales_qty=("sales_qty", "sum"),
        sales_rub=("sales_rub", "sum"),
        stock_end=("stock", "last"),               # сток на конец сезона
        order_qty=("order_qty", "sum"),
        weeks_observed=("iso_week", "nunique"),     # сколько недель наблюдалось
        avg_stockout_rate=("stockout_rate", "mean"),  # средняя доля SKU в дефиците
    )
)

print(f"Агрегированный датасет (сезон): {df_season.shape[0]:,} строк")
print(f"Сезонов: {sorted(df_season['season'].unique())}")
print(f"\nРаспределение по сезонам:")
print(df_season.groupby("season").agg(
    магазинов=("store", "nunique"),
    подгрупп=("subgroup_key", "nunique"),
    строк=("sales_qty", "count"),
).to_string())

Агрегированный датасет (сезон): 24,570 строк
Сезонов: [np.int64(2024), np.int64(2025)]

Распределение по сезонам:
        магазинов  подгрупп  строк
season                            
2024          142        80  10869
2025          172        83  13701


In [29]:
# Подблок 5.6 — Коэффициент роста сети (октябрь 2025 vs октябрь 2024)
# Наивный бейзлайн прогноза строится как «факт 2024 × коэффициент роста», коэффициент должен быть рассчитан по данным, доступным до начала сезона (точка отсечки — 15 октября), поэтому берется только октябрьские недели, по сопоставимым магазинам (тем, которые присутствуют в обоих сезонах)
october = df_week[df_week["period"] == "october"]
# сопоставимые магазины — те, что есть в обоих сезонах
stores_2024 = set(october.loc[october["season"] == 2024, "store"].unique())
stores_2025 = set(october.loc[october["season"] == 2025, "store"].unique())
comparable_stores = stores_2024 & stores_2025
oct_comparable = october[october["store"].isin(comparable_stores)]
oct_totals = oct_comparable.groupby("season")["sales_qty"].sum()
growth_coeff = oct_totals[2025] / oct_totals[2024]
print(f"Сопоставимых магазинов (есть в обоих октябрях): {len(comparable_stores)}")
print(f"Продажи октябрь 2024: {oct_totals[2024]:,.0f} шт")
print(f"Продажи октябрь 2025: {oct_totals[2025]:,.0f} шт")
print(f"Коэффициент роста: {growth_coeff:.3f}")

Сопоставимых магазинов (есть в обоих октябрях): 134
Продажи октябрь 2024: 66,813 шт
Продажи октябрь 2025: 63,169 шт
Коэффициент роста: 0.945


In [30]:
# Подблок 5.7 — Производные атрибуты магазина: возраст и признак «новый» (магазин считается «новым», если он открылся после 1 сентября года, предшествующего сезону — то есть у него не было полного прошлого НГ-сезона для сравнения, новые магазины не могут участвовать в наивном бейзлайне (нет факта за прошлый год), но могут быть hold-out выборкой для проверки прогноза по атрибутам)
season_start = df_season["season"].map({2024: pd.Timestamp("2024-10-01"),
                                         2025: pd.Timestamp("2025-10-01")})

df_season["store_age_days"] = (season_start - df_season["open_date"]).dt.days

# «новый» = открыт после 1 сентября предыдущего года
new_cutoff = df_season["season"].map({2024: pd.Timestamp("2024-09-01"),
                                       2025: pd.Timestamp("2025-09-01")})
df_season["is_new_store"] = df_season["open_date"] >= new_cutoff

new_counts = df_season.groupby("season")["is_new_store"].sum()
total_counts = df_season.groupby("season")["store"].nunique()

print("Новых магазинов (без полного прошлого сезона):")
for s in sorted(df_season["season"].unique()):
    print(f"  Сезон {s}: {int(new_counts[s])} подгруппо-магазинов "
          f"({int(new_counts[s] / df_season['subgroup_key'].nunique())} магазинов)")

Новых магазинов (без полного прошлого сезона):
  Сезон 2024: 1807 подгруппо-магазинов (21 магазинов)
  Сезон 2025: 925 подгруппо-магазинов (11 магазинов)


In [31]:
# Подблок 5.8 — Сохранение промежуточных результатов
# Сохранение три датасета:
# - `df_week` — понедельная агрегация (для EDA, графиков динамики, определения пика)
# - `df_season` — сезонная агрегация (для прогноза и бейзлайнов)
# - `store_ref` — справочник магазинов (для merge в визуализациях)
df_week.to_parquet(PROCESSED_DIR / "df_week.parquet", index=False)
df_season.to_parquet(PROCESSED_DIR / "df_season.parquet", index=False)
store_ref.to_parquet(PROCESSED_DIR / "store_ref.parquet", index=False)

print("Сохранено в data/processed/:")
print(f"  df_week.parquet    — {df_week.shape[0]:,} строк (неделя)")
print(f"  df_season.parquet  — {df_season.shape[0]:,} строк (сезон)")
print(f"  store_ref.parquet  — {store_ref.shape[0]} строк (справочник)")
print(f"\nКоэффициент роста сети (октябрь): {growth_coeff:.3f}")

Сохранено в data/processed/:
  df_week.parquet    — 392,351 строк (неделя)
  df_season.parquet  — 24,570 строк (сезон)
  store_ref.parquet  — 172 строк (справочник)

Коэффициент роста сети (октябрь): 0.945


**Выводы по Блоку 5:**

Что сделано и почему:
- **`ever_active`** — отделение реального ассортимента от пар «магазин × SKU», которых не должно было быть. Без этого оценка дефицита была бы завышена.
- **`is_stockout`** — зафиксирован дефицит как нулевой остаток у активной пары. Определение опирается на факт (сток = 0), а не на прокси (остатки < продажи), что методологически надёжнее.
- **Функциональные окна** — октябрь, основной сезон и январь разделены по ролям: признаки, целевая переменная, оценка неликвида.
- **Агрегация до подгруппы** — единица анализа повышена до уровня, на котором принимается решение о распределении.
- **Коэффициент роста** — рассчитан по сопоставимым магазинам и по данным, доступным ДО начала сезона (без утечки данных).
- **Признак «новый магазин»** — выделены точки, у которых нет полного прошлого сезона для наивного прогноза.